# Creating a DB in sqlite3


## SQL in sqlite3

In [1]:
# # @title Timer for 'h' hours
# from datetime import datetime, timezone
# from time import sleep
# import requests
# import os
# from google.colab import output
# from google.colab import userdata

In [2]:
# h = 10    # number of hours
# print("Timer 2")
# print(f"Started at ... \n{datetime.now(timezone.utc).timetz().isoformat(timespec='seconds')}")

# for i in range(60*60*h):
#   utc_now = datetime.now(timezone.utc)
#   print(f"\r{utc_now.timetz().isoformat(timespec='seconds')} {'==' * (utc_now.second % 10)}", end='')
#   sleep(1)

Using Jupyter magics

In [3]:
# Capture without a variable sends stdout and stderr to /dev/null
%%capture
%%bash
apt-get update
apt-get install -y sqlite3 wamerican-insane


In [4]:
!sqlite3 --help
!sqlite3 --version

Usage: sqlite3 [OPTIONS] [FILENAME [SQL]]
FILENAME is the name of an SQLite database. A new database is created
if the file does not previously exist. Defaults to :memory:.
OPTIONS include:
   --                   treat no subsequent arguments as options
   -A ARGS...           run ".archive ARGS" and exit
   -append              append the database to the end of the file
   -ascii               set output mode to 'ascii'
   -bail                stop after hitting an error
   -batch               force batch I/O
   -box                 set output mode to 'box'
   -column              set output mode to 'column'
   -cmd COMMAND         run "COMMAND" before reading stdin
   -csv                 set output mode to 'csv'
   -deserialize         open the database using sqlite3_deserialize()
   -echo                print inputs before execution
   -init FILENAME       read/process named file
   -[no]header          turn headers on or off
   -help                show this message
   -html    

In [5]:
!ls -l /usr/share/dict/


total 6768
-rw-r--r-- 1 root root 6922426 Jan 20  2022 american-english-insane
-rw-r--r-- 1 root root     199 Sep  7  2023 README.select-wordlist
lrwxrwxrwx 1 root root      30 Nov  9 21:11 words -> /etc/dictionaries-common/words


In [6]:
!wc -l /usr/share/dict/american-english-insane


663473 /usr/share/dict/american-english-insane


In [7]:
%%bash
egrep '^[a-z]{5}$' /usr/share/dict/american-english-insane | wc -l

17516


In [8]:
%%bash
egrep '^[a-z]{5}$' /usr/share/dict/american-english-insane |
egrep '^...u.$' | wc -l

704


In [19]:
%%bash
egrep '^[a-z]{5}$' /usr/share/dict/american-english-insane |
# egrep '^...u.$' | wc -l
egrep '^..a.y$' | egrep -v '[iuphczdfkt]'


agamy
alary
araby
beamy
beany
braxy
glary
goary
gramy
gravy
leany
leary
leavy
loamy
mealy
meany
orary
ovary
reamy
reasy
roary
sealy
seamy
seary
seavy
snary
soary
swaly
swamy
vealy
weary


In [20]:
%%bash
{
  echo "blah blah blah"
  echo
  echo
  echo words{1..7} | tr ' ' ,
  egrep '^[a-z]{5}$' /usr/share/dict/words |
    shuf -n 420 |
    paste -d, - - - - - - |
    cat -n |
    tr -d ' ' |
    tr '\t' ,
} > /tmp/words.5.csv


In [21]:
# Show the first few entries
!head /tmp/words.5.csv | cat -nvet


     1	blah blah blah$
     2	$
     3	$
     4	words1,words2,words3,words4,words5,words6,words7$
     5	1,roded,sours,frase,tules,badan,sawed$
     6	2,khuen,axiom,bacca,hater,nites,nisan$
     7	3,mbira,luges,erugo,koine,flary,sophy$
     8	4,dunes,hello,nuddy,braca,zambo,would$
     9	5,jibes,betty,holey,yukon,polls,weids$
    10	6,ripal,sagos,decem,bedes,atole,slait$


In [22]:
!ls -la /tmp/words.5.csv


-rw-r--r-- 1 root root 2788 Nov  9 21:33 /tmp/words.5.csv


In [23]:
# Count the number of lines, words, characters
!wc /tmp/words.5.csv


  74   74 2788 /tmp/words.5.csv


In [24]:
# Remove any existing database
!rm -rf words.sql3


In [25]:
# Import CSV into SQLite3
%%script sqlite3 --column --header words.sql3
.mode csv
.import '| grep -A 100000000 ^words /tmp/words.5.csv' words


In [26]:
# Show that the database now exists
!ls -l


total 12
drwxr-xr-x 1 root root 4096 Nov  6 16:29 sample_data
-rw-r--r-- 1 root root 8192 Nov  9 21:34 words.sql3


In [27]:
# Query the table
%%script sqlite3 --column --header words.sql3
select *
from words
limit 10


words1  words2  words3  words4  words5  words6  words7
------  ------  ------  ------  ------  ------  ------
1       roded   sours   frase   tules   badan   sawed 
2       khuen   axiom   bacca   hater   nites   nisan 
3       mbira   luges   erugo   koine   flary   sophy 
4       dunes   hello   nuddy   braca   zambo   would 
5       jibes   betty   holey   yukon   polls   weids 
6       ripal   sagos   decem   bedes   atole   slait 
7       foyle   clape   hylic   awide   swell   riden 
8       pilau   orach   sudsy   pridy   seres   firth 
9       romps   swonk   cover   rowed   terse   rojis 
10      whute   guiro   geest   carps   smore   kiaki 


In [28]:
# View schema
%%script sqlite3 --column --header words.sql3
.schema


CREATE TABLE IF NOT EXISTS "words"(
"words1" TEXT, "words2" TEXT, "words3" TEXT, "words4" TEXT,
 "words5" TEXT, "words6" TEXT, "words7" TEXT);


In [29]:
# Create a table with correct data types, including a primary key ( unique, indexed column )
%%script sqlite3 --column --header words.sql3
CREATE TABLE words_real(
  "words1" INTEGER PRIMARY KEY ASC,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);


In [30]:
# View schema
%%script sqlite3 --column --header words.sql3
.schema


CREATE TABLE IF NOT EXISTS "words"(
"words1" TEXT, "words2" TEXT, "words3" TEXT, "words4" TEXT,
 "words5" TEXT, "words6" TEXT, "words7" TEXT);
CREATE TABLE words_real(
  "words1" INTEGER PRIMARY KEY ASC,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);


In [31]:
# Select data into correct table, i.e. "copy" the data
%%script sqlite3 --column --header words.sql3
INSERT INTO words_real
SELECT *
FROM words


In [32]:
# Show the schema and the first 10 entries
%%script sqlite3 --column --header words.sql3
.schema
select * from words_real limit 10 ;


CREATE TABLE IF NOT EXISTS "words"(
"words1" TEXT, "words2" TEXT, "words3" TEXT, "words4" TEXT,
 "words5" TEXT, "words6" TEXT, "words7" TEXT);
CREATE TABLE words_real(
  "words1" INTEGER PRIMARY KEY ASC,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);
words1  words2  words3  words4  words5  words6  words7
------  ------  ------  ------  ------  ------  ------
1       roded   sours   frase   tules   badan   sawed 
2       khuen   axiom   bacca   hater   nites   nisan 
3       mbira   luges   erugo   koine   flary   sophy 
4       dunes   hello   nuddy   braca   zambo   would 
5       jibes   betty   holey   yukon   polls   weids 
6       ripal   sagos   decem   bedes   atole   slait 
7       foyle   clape   hylic   awide   swell   riden 
8       pilau   orach   sudsy   pridy   seres   firth 
9       romps   swonk   cover   rowed   terse   rojis 
10      whute   guiro   geest   carps   smore   kiaki 


In [33]:
# Show the schema and the first 10 entries
%%script sqlite3 --column --header words.sql3
.schema
select * from words limit 10 ;
select * from words_real limit 10 ;


CREATE TABLE IF NOT EXISTS "words"(
"words1" TEXT, "words2" TEXT, "words3" TEXT, "words4" TEXT,
 "words5" TEXT, "words6" TEXT, "words7" TEXT);
CREATE TABLE words_real(
  "words1" INTEGER PRIMARY KEY ASC,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);
words1  words2  words3  words4  words5  words6  words7
------  ------  ------  ------  ------  ------  ------
1       roded   sours   frase   tules   badan   sawed 
2       khuen   axiom   bacca   hater   nites   nisan 
3       mbira   luges   erugo   koine   flary   sophy 
4       dunes   hello   nuddy   braca   zambo   would 
5       jibes   betty   holey   yukon   polls   weids 
6       ripal   sagos   decem   bedes   atole   slait 
7       foyle   clape   hylic   awide   swell   riden 
8       pilau   orach   sudsy   pridy   seres   firth 
9       romps   swonk   cover   rowed   terse   rojis 
10      whute   guiro   geest   carps   smore   kiaki 
words1  words2  words3  words4  

In [34]:
# Create an index on the last column
%%script sqlite3 --column --header words.sql3
create index if not exists words7 on words_real (words7) ;


In [35]:
# Show the schema, noting the index
%%script sqlite3 --column --header words.sql3
.schema


CREATE TABLE IF NOT EXISTS "words"(
"words1" TEXT, "words2" TEXT, "words3" TEXT, "words4" TEXT,
 "words5" TEXT, "words6" TEXT, "words7" TEXT);
CREATE TABLE words_real(
  "words1" INTEGER PRIMARY KEY ASC,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);
CREATE INDEX words7 on words_real (words7) ;


In [36]:
# Create a derived column
%%script sqlite3 --column --header words.sql3
select *, words6 || "+" || words7 as "both"
from words_real
limit 10


words1  words2  words3  words4  words5  words6  words7  both       
------  ------  ------  ------  ------  ------  ------  -----------
1       roded   sours   frase   tules   badan   sawed   badan+sawed
2       khuen   axiom   bacca   hater   nites   nisan   nites+nisan
3       mbira   luges   erugo   koine   flary   sophy   flary+sophy
4       dunes   hello   nuddy   braca   zambo   would   zambo+would
5       jibes   betty   holey   yukon   polls   weids   polls+weids
6       ripal   sagos   decem   bedes   atole   slait   atole+slait
7       foyle   clape   hylic   awide   swell   riden   swell+riden
8       pilau   orach   sudsy   pridy   seres   firth   seres+firth
9       romps   swonk   cover   rowed   terse   rojis   terse+rojis
10      whute   guiro   geest   carps   smore   kiaki   smore+kiaki


## Using pandas and sqlite3 Python modules

In [37]:
import sqlite3 as db
import pandas as pd


#### Using a context manager

The context manager automatically closes the connection when finished using the database.


In [38]:
# Connect to a sqlite database and load a result set into a data frame
with db.connect("words.sql3") as db_con:
  words = pd.read_sql_query("select * from words", db_con)
words


,words1,words2,words3,words4,words5,words6,words7
0,1,roded,sours,frase,tules,badan,sawed
1,2,khuen,axiom,bacca,hater,nites,nisan
2,3,mbira,luges,erugo,koine,flary,sophy
3,4,dunes,hello,nuddy,braca,zambo,would
4,5,jibes,betty,holey,yukon,polls,weids
...,...,...,...,...,...,...,...
65,66,barye,coins,gusle,strue,laids,mamry
66,67,elfin,boars,apted,husky,meath,named
67,68,treey,chays,nonya,eager,swiss,pling
68,69,armed,tawer,melch,bagre,chink,mosks


#### Without using a context manager



The connection to the database stays open until you explicitly close it.

In [39]:
# Connect to a sqlite database and load a result set into a data frame
db_con = db.connect("words.sql3")
words = pd.read_sql_query("select * from words", db_con)
words


,words1,words2,words3,words4,words5,words6,words7
0,1,roded,sours,frase,tules,badan,sawed
1,2,khuen,axiom,bacca,hater,nites,nisan
2,3,mbira,luges,erugo,koine,flary,sophy
3,4,dunes,hello,nuddy,braca,zambo,would
4,5,jibes,betty,holey,yukon,polls,weids
...,...,...,...,...,...,...,...
65,66,barye,coins,gusle,strue,laids,mamry
66,67,elfin,boars,apted,husky,meath,named
67,68,treey,chays,nonya,eager,swiss,pling
68,69,armed,tawer,melch,bagre,chink,mosks


In [40]:
# Show some info about the data frame
words.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   words1  70 non-null     object
 1   words2  70 non-null     object
 2   words3  70 non-null     object
 3   words4  70 non-null     object
 4   words5  70 non-null     object
 5   words6  70 non-null     object
 6   words7  70 non-null     object
dtypes: object(7)
memory usage: 4.0+ KB


In [41]:
# Query the database and load the result into a data frame
words_real = pd.read_sql_query("select * from words_real", db_con)
words_real


,words1,words2,words3,words4,words5,words6,words7
0,1,roded,sours,frase,tules,badan,sawed
1,2,khuen,axiom,bacca,hater,nites,nisan
2,3,mbira,luges,erugo,koine,flary,sophy
3,4,dunes,hello,nuddy,braca,zambo,would
4,5,jibes,betty,holey,yukon,polls,weids
...,...,...,...,...,...,...,...
65,66,barye,coins,gusle,strue,laids,mamry
66,67,elfin,boars,apted,husky,meath,named
67,68,treey,chays,nonya,eager,swiss,pling
68,69,armed,tawer,melch,bagre,chink,mosks


In [42]:
# Show some info about the data frame
words_real.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   words1  70 non-null     int64 
 1   words2  70 non-null     object
 2   words3  70 non-null     object
 3   words4  70 non-null     object
 4   words5  70 non-null     object
 5   words6  70 non-null     object
 6   words7  70 non-null     object
dtypes: int64(1), object(6)
memory usage: 4.0+ KB


In [43]:
# Show the first 10 entries in the new data frame
words_real2 = words_real[:10]
words_real2


,words1,words2,words3,words4,words5,words6,words7
0,1,roded,sours,frase,tules,badan,sawed
1,2,khuen,axiom,bacca,hater,nites,nisan
2,3,mbira,luges,erugo,koine,flary,sophy
3,4,dunes,hello,nuddy,braca,zambo,would
4,5,jibes,betty,holey,yukon,polls,weids
5,6,ripal,sagos,decem,bedes,atole,slait
6,7,foyle,clape,hylic,awide,swell,riden
7,8,pilau,orach,sudsy,pridy,seres,firth
8,9,romps,swonk,cover,rowed,terse,rojis
9,10,whute,guiro,geest,carps,smore,kiaki


In [44]:
# Save the entire dataframe to a new table in the existing database
words_real2.to_sql("words_real2", db_con, if_exists="replace")


10

In [45]:
# Show the schema of the entire database
%%script sqlite3 --column --header words.sql3
.schema


CREATE TABLE IF NOT EXISTS "words"(
"words1" TEXT, "words2" TEXT, "words3" TEXT, "words4" TEXT,
 "words5" TEXT, "words6" TEXT, "words7" TEXT);
CREATE TABLE words_real(
  "words1" INTEGER PRIMARY KEY ASC,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);
CREATE INDEX words7 on words_real (words7) ;
CREATE TABLE IF NOT EXISTS "words_real2" (
"index" INTEGER,
  "words1" INTEGER,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);
CREATE INDEX "ix_words_real2_index"ON "words_real2" ("index");


Notice that we didn't close the connection.


In [46]:
# the right thing to do is to close the connection
db_con.close()


In [47]:
!ls -l


total 32
drwxr-xr-x 1 root root  4096 Nov  6 16:29 sample_data
-rw-r--r-- 1 root root 24576 Nov  9 21:51 words.sql3


In [ ]:
# close the db! (or use a db manager)